# Forced alignment demo

Align one short audio clip with its known transcript and print the start and end time of every word.

Tool: [stable-ts](https://github.com/jianfch/stable-ts) (`model.align`). Input: `data/sample.wav` and `data/sample.txt`.

**Status:** written but not yet executed. The stable-ts API calls below may need small fixes.

In [ ]:
# Run this notebook inside the conda env: conda env create -f environment.yml
import shutil
import stable_whisper
print('ffmpeg:', shutil.which('ffmpeg'))

In [ ]:
from pathlib import Path

AUDIO = Path('data/sample.wav')
TEXT = Path('data/sample.txt')
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)

assert AUDIO.exists(), f'Missing {AUDIO}; see data/SOURCES.md'
assert TEXT.exists(), f'Missing {TEXT}; see data/SOURCES.md'

transcript = TEXT.read_text(encoding='utf-8').strip()
print(transcript)

In [ ]:
import stable_whisper

model = stable_whisper.load_model('base')  # small model; runs on CPU
result = model.align(str(AUDIO), transcript, language='en')

In [ ]:
import pandas as pd

rows = [
    {'word': w.word.strip(), 'start': round(w.start, 2), 'end': round(w.end, 2)}
    for seg in result.segments
    for w in seg.words
]
df = pd.DataFrame(rows)
df

In [ ]:
# Sanity checks
n_transcript_words = len(transcript.split())
print('words in transcript:', n_transcript_words)
print('words aligned:      ', len(df))
print('last word ends at:  ', df['end'].max(), 'seconds')
print('start times increasing:', df['start'].is_monotonic_increasing)

In [ ]:
# Save outputs
df.to_csv(OUT / 'words.csv', index=False)
result.save_as_json(str(OUT / 'alignment.json'))
result.to_srt_vtt(str(OUT / 'alignment.srt'), word_level=True)
print('saved to', OUT.resolve())